In [10]:
import cv2
import numpy as np
imagen = cv2.imread("img/mano.jpeg", cv2.IMREAD_GRAYSCALE)

In [11]:
imagen_suave = cv2.GaussianBlur(imagen, (5,5), 0)

_, imagen_bin = cv2.threshold(imagen_suave, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

imagen_bin = cv2.bitwise_not(imagen_bin)

In [12]:
kernel = np.ones((3,3), np.uint8)
imagen_bin = cv2.morphologyEx(imagen_bin, cv2.MORPH_OPEN, kernel)
imagen_bin = cv2.morphologyEx(imagen_bin, cv2.MORPH_CLOSE, kernel)

In [13]:
skeleton = np.zeros(imagen_bin.shape, np.uint8)

elemento = cv2.getStructuringElement(cv2.MORPH_CROSS, (3,3))
img_temp = imagen_bin.copy()

while True:

    eroded = cv2.erode(img_temp, elemento)
    opened = cv2.morphologyEx(eroded, cv2.MORPH_OPEN, elemento)

    temp = cv2.subtract(eroded, opened)
    skeleton = cv2.bitwise_or(skeleton, temp)

    img_temp = eroded.copy()

    if cv2.countNonZero(img_temp) == 0:
        break


In [15]:
# Convertir a BGR para poder unirlas
img_original_color = cv2.cvtColor(imagen, cv2.COLOR_GRAY2BGR)
img_bin_color = cv2.cvtColor(imagen_bin, cv2.COLOR_GRAY2BGR)
img_skeleton_color = cv2.cvtColor(skeleton, cv2.COLOR_GRAY2BGR)

# Unir las imágenes horizontalmente
imagen_compuesta = np.hstack((img_original_color, img_bin_color, img_skeleton_color))

# Mostrar resultado
cv2.imshow("Original - Binaria - Esqueleto", imagen_compuesta)

cv2.waitKey(0)
cv2.destroyAllWindows()